# Evaluate RTLE Policy Performance

## Overview
This notebook evaluates trained RTLE policies across different market conditions and configurations.
It loads pre-trained models and runs them in deterministic mode for consistent evaluation.

## Cell 1: Imports and Setup

In [ ]:
from dataclasses import dataclass
import gymnasium as gym
import torch
import numpy as np
import time
import os
import sys

notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from simulation.market_gym import Market
from rl_files.actor_critic import AgentLogisticNormal, DirichletAgent, Agent

print(f"Project root: {project_root}")
print("All imports successful")

## Cell 2: Evaluation Configuration

In [ ]:
@dataclass
class EvalConfig:
    model_path: str = "./models"
    reward_output_dir: str = "./rewards"
    n_eval_episodes: int = 10000
    deterministic_action: bool = True
    env_types: list = None
    num_lots: list = None
    eval_seed: int = 100
    exp_names: list = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    
    def __post_init__(self):
        if self.env_types is None:
            self.env_types = ['noise', 'flow', 'strategic']
        if self.num_lots is None:
            self.num_lots = [20, 60]
        if self.exp_names is None:
            self.exp_names = ['log_normal']

eval_config = EvalConfig(
    n_eval_episodes=100,
    deterministic_action=True,
    env_types=['noise', 'flow', 'strategic'],
    num_lots=[20, 60],
    exp_names=['log_normal']
)

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    device = torch.device(f"cuda:{num_gpus - 1}")
    print(f"Using GPU: cuda:{num_gpus - 1}")
else:
    device = torch.device(eval_config.device)
    print(f"Using device: {device}")

## Cell 3: Helper Functions

In [ ]:
def make_env(config):
    def thunk():
        return Market(config)
    return thunk

def load_agent(agent_type, envs, device, model_path=None):
    if agent_type == 'log_normal':
        agent = AgentLogisticNormal(envs, variance_scaling=True).to(device)
    elif agent_type == 'log_normal_learn_std':
        agent = AgentLogisticNormal(envs, variance_scaling=False).to(device)
    elif agent_type == 'dirichlet':
        agent = DirichletAgent(envs).to(device)
    elif agent_type == 'normal':
        agent = Agent(envs).to(device)
    else:
        raise ValueError(f"Unknown agent type: {agent_type}")
    
    if model_path and os.path.exists(model_path):
        state_dict = torch.load(model_path, map_location=device)
        agent.load_state_dict(state_dict)
        print(f"Loaded model: {os.path.basename(model_path)}")
    
    agent.eval()
    return agent

print("Helper functions defined")

## Cell 4: Evaluation Function

In [ ]:
def evaluate_model(
    model_path, agent_type, env_type, num_lots,
    n_eval_episodes, eval_seed, device,
    deterministic=True, num_envs=128, drop_feature=None
):
    print(f"Evaluating {os.path.basename(model_path)} ({agent_type}, {env_type}, {num_lots} lots)")
    
    configs = [{
        'market_env': env_type,
        'execution_agent': 'rl_agent',
        'volume': num_lots,
        'seed': eval_seed + s,
        'terminal_time': 150,
        'time_delta': 15,
        'drop_feature': drop_feature
    } for s in range(num_envs)]
    
    env_fns = [make_env(config) for config in configs]
    envs = gym.vector.AsyncVectorEnv(env_fns=env_fns)
    agent = load_agent(agent_type, envs, device, model_path)
    
    obs, _ = envs.reset()
    episodic_returns = []
    start_time = time.time()
    
    print(f"  Running {n_eval_episodes} episodes...")
    with torch.no_grad():
        obs_gpu = torch.Tensor(obs).to(device)
        while len(episodic_returns) < n_eval_episodes:
            if deterministic and hasattr(agent, 'deterministic_action'):
                actions = agent.deterministic_action(obs_gpu)
            else:
                actions, _, _, _ = agent.get_action_and_value(obs_gpu)
            
            next_obs, _, _, _, infos = envs.step(actions.cpu().numpy())
            obs_gpu = torch.Tensor(next_obs).to(device)
            
            if "final_info" in infos:
                for info in infos["final_info"]:
                    if info is not None:
                        episodic_returns.append(info['reward'])
    
    eval_time = time.time() - start_time
    envs.close()
    
    rewards = np.array(episodic_returns)
    stats = {
        'mean': np.mean(rewards),
        'std': np.std(rewards),
        'min': np.min(rewards),
        'max': np.max(rewards),
        'count': len(rewards),
        'time': eval_time
    }
    
    print(f"  Mean: {stats['mean']:.4f}, Std: {stats['std']:.4f}, Time: {eval_time:.2f}s")
    return rewards, stats

print("Evaluation function defined")

## Cell 5: Batch Evaluation

In [ ]:
os.makedirs(eval_config.reward_output_dir, exist_ok=True)

train_seed = 0
num_iterations = 400
batch_size = 12800
all_results = []

for exp_name in eval_config.exp_names:
    for env_type in eval_config.env_types:
        for num_lots in eval_config.num_lots:
            print(f"\nEvaluating {exp_name} - {env_type} - {num_lots} lots")
            
            run_name = (f"{env_type}_{num_lots}_seed_{train_seed}_eval_seed_{eval_config.eval_seed}_"
                        f"eval_episodes_{eval_config.n_eval_episodes}_num_iterations_{num_iterations}_"
                        f"bsize_{batch_size}_{exp_name}")
            
            model_path = os.path.join(eval_config.model_path, f"{run_name}.pt")
            
            if not os.path.exists(model_path):
                print(f"  Model not found: {model_path}")
                continue
            
            try:
                rewards, stats = evaluate_model(
                    model_path=model_path,
                    agent_type=exp_name,
                    env_type=env_type,
                    num_lots=num_lots,
                    n_eval_episodes=eval_config.n_eval_episodes,
                    eval_seed=eval_config.eval_seed,
                    device=device,
                    deterministic=eval_config.deterministic_action,
                    num_envs=128
                )
                
                saving_tag = '_deterministic_action' if eval_config.deterministic_action else '_stochastic'
                output_file = os.path.join(
                    eval_config.reward_output_dir,
                    f"{run_name}{saving_tag}.npz"
                )
                np.savez(output_file, rewards=rewards)
                print(f"  Saved to {os.path.basename(output_file)}")
                
                all_results.append({
                    'exp_name': exp_name,
                    'env_type': env_type,
                    'num_lots': num_lots,
                    'stats': stats
                })
                
            except Exception as e:
                print(f"  Error: {e}")

## Cell 6: Results Summary

In [ ]:
print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)
print(f"Total models evaluated: {len(all_results)}")
print(f"Deterministic: {eval_config.deterministic_action}")
print(f"Episodes per model: {eval_config.n_eval_episodes}")

if all_results:
    print(f"\n{'Experiment':<20} {'Environment':<15} {'Lots':<6} {'Mean':<12} {'Std':<10}")
    print("-" * 70)
    
    for result in all_results:
        stats = result['stats']
        print(f"{result['exp_name']:<20} {result['env_type']:<15} {result['num_lots']:<6} "
              f"{stats['mean']:>11.4f} {stats['std']:>9.4f}")
    
    if len(all_results) > 0:
        best_idx = np.argmax([r['stats']['mean'] for r in all_results])
        worst_idx = np.argmin([r['stats']['mean'] for r in all_results])
        best = all_results[best_idx]
        worst = all_results[worst_idx]
        
        print("\n" + "-" * 70)
        print(f"Best:  {best['exp_name']} | {best['env_type']} | {best['num_lots']} lots | Mean={best['stats']['mean']:.4f}")
        print(f"Worst: {worst['exp_name']} | {worst['env_type']} | {worst['num_lots']} lots | Mean={worst['stats']['mean']:.4f}")
else:
    print("No results available")

print("="*70)